# Actor function category: validation on matched actors

Run after the manual checks. This evaluates how well gpt-oss assigns the **actor
function category** (a, b, c, d), **only for actors that both the human coder and
gpt-oss identified** (the matched set). Actors coded by only one side are excluded,
because a function label can only be compared when both sides labelled the actor.

Matched actors are paired through the `matched_with` column, and the human label is
treated as the reference. Per language it reports:

* **n_matched** actors compared, and **accuracy** (share with the same category)
* **Cohen's kappa** (chance-corrected agreement) and **macro-F1**
* a per-category precision / recall / F1 table and a human x gpt confusion matrix

Results are written to `actor_function_results.xlsx`.

In [ ]:
import pandas as pd
from sklearn.metrics import (accuracy_score, cohen_kappa_score, f1_score,
                             classification_report, confusion_matrix)

FILES = {
    "English": ["human_gpt_comparison_actors_english_UK.xlsx",
                "human_gpt_comparison_actors_english_IE.xlsx"],
    "Dutch":   ["human_gpt_comparison_actors_dutch.xlsx"],
    "Turkish": ["human_gpt_comparison_actors_turkish.xlsx"],
}
CATEGORIES = ["a", "b", "c", "d"]

In [ ]:
def matched_pairs(paths):
    """Return (human_func, gpt_func) for actors coded by BOTH sides."""
    df = pd.concat([pd.read_excel(p, sheet_name="comparison") for p in paths],
                   ignore_index=True)
    df["missedbygpt"] = pd.to_numeric(df["missedbygpt"], errors="coerce")
    df["coded extra"] = pd.to_numeric(df["coded extra"], errors="coerce")

    # matched human actors = human rows gpt also coded (missedbygpt == 0)
    hum = df[(df["coder"] == "human") & (df["missedbygpt"] == 0)][
        ["article_id", "actor_function", "matched_with"]
    ].rename(columns={"actor_function": "human_func"})

    # only matched gpt actors (coded extra == 0) are valid counterparts
    gpt = df[(df["coder"] == "gpt_oss") & (df["coded extra"] == 0)][
        ["article_id", "actor_name", "actor_function"]
    ].rename(columns={"actor_name": "gpt_name", "actor_function": "gpt_func"})

    # pair each matched human actor with its gpt counterpart (by name, within article)
    pairs = hum.merge(gpt, left_on=["article_id", "matched_with"],
                      right_on=["article_id", "gpt_name"], how="left")

    n_matched = len(hum)
    pairs = pairs.dropna(subset=["human_func", "gpt_func"])
    return pairs[["human_func", "gpt_func"]], n_matched

In [ ]:
rows = []
pairs_by_lang = {}
for lang, paths in FILES.items():
    pairs, n_matched = matched_pairs(paths)
    pairs_by_lang[lang] = pairs
    y_true, y_pred = pairs["human_func"], pairs["gpt_func"]
    rows.append({
        "n_matched":   n_matched,
        "n_compared":  len(pairs),                       # matched actors with a category on both sides
        "accuracy":    round(accuracy_score(y_true, y_pred), 2),
        "kappa":       round(cohen_kappa_score(y_true, y_pred, labels=CATEGORIES), 2),
        "macro_F1":    round(f1_score(y_true, y_pred, labels=CATEGORIES, average="macro"), 2),
        "weighted_F1": round(f1_score(y_true, y_pred, labels=CATEGORIES, average="weighted"), 2),
    })

summary = pd.DataFrame(rows, index=FILES.keys())
summary.index.name = "Language"
summary

In [ ]:
# per-category precision/recall/F1 and confusion matrix, per language
reports, confusions = {}, {}
for lang, pairs in pairs_by_lang.items():
    y_true, y_pred = pairs["human_func"], pairs["gpt_func"]
    rep = pd.DataFrame(classification_report(
        y_true, y_pred, labels=CATEGORIES, output_dict=True, zero_division=0
    )).T.round(2)
    cm = pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=CATEGORIES),
        index=[f"human_{c}" for c in CATEGORIES],
        columns=[f"gpt_{c}" for c in CATEGORIES],
    )
    reports[lang], confusions[lang] = rep, cm
    print(f"\n===== {lang} =====")
    print(rep.to_string())
    print("\nconfusion (rows = human, cols = gpt):")
    print(cm.to_string())

In [ ]:
# write everything to one Excel (summary + per-language detail)
with pd.ExcelWriter("actor_function_results.xlsx", engine="openpyxl") as xl:
    summary.to_excel(xl, sheet_name="summary")
    for lang in FILES:
        reports[lang].to_excel(xl, sheet_name=f"{lang}_byclass")
        confusions[lang].to_excel(xl, sheet_name=f"{lang}_confusion")
print("saved -> actor_function_results.xlsx")